<a href="https://colab.research.google.com/github/Zulabat/My-Python-Library/blob/main/Homework_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('https://raw.githubusercontent.com/amandeep0/IS451/main/data/loans.csv')
df.head()

,CreditPolicy,Purpose,IntRate,Installment,LogAnnualInc,Dti,Fico,DaysWithCrLine,RevolBal,RevolUtil,InqLast6mths,Delinq2yrs,PubRec,NotFullyPaid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0


a.i

In [3]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.3, random_state=42)

a.ii

In [4]:
from sklearn.model_selection import train_test_split
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score

# Baseline prediction: assume all loans are paid back (NotFullyPaid = 0)
y_pred_baseline = [0] * len(df_test)

# Actual target variable
y_true = df_test["NotFullyPaid"]

# Calculate the accuracy of the baseline model
accuracy_baseline = accuracy_score(y_true, y_pred_baseline)
print("Baseline model accuracy (assuming all loans are fully paid):", accuracy_baseline)

Baseline model accuracy (assuming all loans are fully paid): 0.83785664578984


In [5]:
import statsmodels.formula.api as smf
logistic_regression_model_1 = smf.logit('NotFullyPaid ~ CreditPolicy + Purpose	+ IntRate	+ Installment	+ LogAnnualInc	+ Dti	+ Fico +	DaysWithCrLine + RevolBal + RevolUtil	+ InqLast6mths + Delinq2yrs + PubRec', data=df_train)
logistic_regression_model_1_results = logistic_regression_model_1.fit()
print(logistic_regression_model_1_results.summary())

Optimization terminated successfully.
         Current function value: 0.409992
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:           NotFullyPaid   No. Observations:                 6704
Model:                          Logit   Df Residuals:                     6685
Method:                           MLE   Df Model:                           18
Date:                Mon, 28 Oct 2024   Pseudo R-squ.:                 0.06453
Time:                        00:21:43   Log-Likelihood:                -2748.6
converged:                       True   LL-Null:                       -2938.2
Covariance Type:            nonrobust   LLR p-value:                 1.989e-69
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                         8.3393      1.545      5.399      0.

CreditPolicy, Installment, LogAnnualInc, Fico, RevolBal , InqLast6mths, Delinq2yrs, PubRec are the independent variable

a.iii

In [7]:
# Get the coefficient of the 'Fico' variable from the model results
fico_coefficient = logistic_regression_model_1_results.params['Fico']

# Calculate the difference in logit values for a 10-point difference in FICO score
logit_difference = -10 * fico_coefficient

print("The value of Logit(A) - Logit(B) for a 10-point difference in FICO score (700 to 710) is:", logit_difference)

The value of Logit(A) - Logit(B) for a 10-point difference in FICO score (700 to 710) is: 0.0900478939342196


a.iv

In [8]:
# Predict probabilities of 'NotFullyPaid' for the test set
df_test['PredictedRisk'] = logistic_regression_model_1_results.predict(df_test)

# Use a threshold of 0.5 to determine predicted classes
df_test['PredictedClass'] = (df_test['PredictedRisk'] >= 0.5).astype(int)

# Calculate the accuracy of the logistic regression model on the test set
logistic_regression_accuracy = accuracy_score(df_test["NotFullyPaid"], df_test["PredictedClass"])

print("Accuracy of the logistic regression model on the test set:", logistic_regression_accuracy)
print("Baseline model accuracy (assuming all loans are fully paid):", accuracy_baseline)

# Comparison
if logistic_regression_accuracy > accuracy_baseline:
    print("The logistic regression model performs better than the baseline model.")
elif logistic_regression_accuracy < accuracy_baseline:
    print("The logistic regression model performs worse than the baseline model.")
else:
    print("The logistic regression model has the same accuracy as the baseline model.")


Accuracy of the logistic regression model on the test set: 0.8382045929018789
Baseline model accuracy (assuming all loans are fully paid): 0.83785664578984
The logistic regression model performs better than the baseline model.


b.i

In [10]:
# Build a logistic regression model with IntRate as the only independent variable
logistic_regression_model_int_rate = smf.logit('NotFullyPaid ~ IntRate', data=df_train)
logistic_regression_model_int_rate_results = logistic_regression_model_int_rate.fit()
print(logistic_regression_model_int_rate_results.summary())

Optimization terminated successfully.
         Current function value: 0.426903
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:           NotFullyPaid   No. Observations:                 6704
Model:                          Logit   Df Residuals:                     6702
Method:                           MLE   Df Model:                            1
Date:                Mon, 28 Oct 2024   Pseudo R-squ.:                 0.02594
Time:                        00:26:27   Log-Likelihood:                -2862.0
converged:                       True   LL-Null:                       -2938.2
Covariance Type:            nonrobust   LLR p-value:                 5.098e-35
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.5983      0.167    -21.607      0.000      -3.925      -3.272
IntRate       15.3201      1.

Interest rate is very significant to detetmine if the loan will be fully paid.

b.ii

In [11]:
# Make predictions on the test set using the fitted model (probabilities)
df_test['PredictedRisk_IntRate'] = logistic_regression_model_int_rate_results.predict(df_test)

# Find the highest predicted probability
max_predicted_risk = df_test['PredictedRisk_IntRate'].max()
print("Highest predicted probability of a loan not being paid back in full:", max_predicted_risk)

# Use a threshold of 0.5 to classify the loans as 'NotFullyPaid' (1 if probability >= 0.5)
df_test['Predicted_NotFullyPaid_IntRate'] = (df_test['PredictedRisk_IntRate'] >= 0.5).astype(int)

# Count how many loans are predicted to not be paid back in full
num_not_fully_paid = df_test['Predicted_NotFullyPaid_IntRate'].sum()
print("Number of loans predicted to not be paid back in full (threshold = 0.5):", num_not_fully_paid)


Highest predicted probability of a loan not being paid back in full: 0.41363952788871816
Number of loans predicted to not be paid back in full (threshold = 0.5): 0


c.i

In [13]:
import math

# Given values
c = 10
r = 0.06
t = 3

# Future Value calculation
future_value = c * math.exp(r * t)
future_value

11.972173631218102

In [14]:
# Given values
c = 10  # initial investment
r = 0.06  # annual interest rate as a proportion
t = 3  # investment period in years

# Future Value calculation
future_value = c * math.exp(r * t)

# Profit calculations
profit_paid_back = future_value - c
profit_not_paid_back = -c

print("Profit if investment is paid back in full:", profit_paid_back)
print("Profit if investment is not paid back in full:", profit_not_paid_back)

Profit if investment is paid back in full: 1.9721736312181015
Profit if investment is not paid back in full: -10


In [15]:
# Define parameters for the calculation
c = 1  # $1 investment
t = 3  # loan period in years

# Calculate profit for each loan in the test set
df_test['Profit'] = np.where(
    df_test['NotFullyPaid'] == 0,  # If loan is fully paid
    c * np.exp(df_test['IntRate'] * t) - c,  # Profit if fully paid back
    -c  # Loss if not paid back
)

# Find the maximum profit in the test set
max_profit = df_test['Profit'].max()
print("Maximum profit of a $1 investment in any loan in the testing set:", max_profit)

Maximum profit of a $1 investment in any loan in the testing set: 0.8894768654675331


In [16]:
# Step 1: Filter for high-interest loans (IntRate >= 15%) in the test set
HighInterest = df_test[df_test['IntRate'] >= 0.15]

# Step 2: Calculate the average profit of a $1 investment in these high-interest loans
average_profit_high_interest = HighInterest['Profit'].mean()
print("Average profit of a $1 investment in high-interest loans:", average_profit_high_interest)

# Step 3: Calculate the proportion of high-interest loans that were not paid back in full
proportion_not_fully_paid = HighInterest['NotFullyPaid'].mean()  # Since NotFullyPaid is 1 for defaulted loans
print("Proportion of high-interest loans not paid back in full:", proportion_not_fully_paid)

Average profit of a $1 investment in high-interest loans: 0.21490899419105775
Proportion of high-interest loans not paid back in full: 0.25721153846153844


In [17]:
# Step 1: Sort the HighInterest loans by PredictedRisk in ascending order
HighInterest_sorted = HighInterest.sort_values(by='PredictedRisk')

# Step 2: Select the 100 loans with the lowest PredictedRisk values
SelectedLoans = HighInterest_sorted.head(100)

# Step 3: Calculate the total profit for an investor who invests $1 in each of these 100 loans
total_profit_selected_loans = SelectedLoans['Profit'].sum()
print("Total profit of an investor who invested $1 in each of the 100 selected loans:", total_profit_selected_loans)

# Step 4: Count how many of the 100 selected loans were not paid back in full
num_not_fully_paid_selected_loans = SelectedLoans['NotFullyPaid'].sum()
print("Number of the 100 selected loans not paid back in full:", num_not_fully_paid_selected_loans)

# Step 5: Compare to the simple strategy
simple_strategy_profit = 20.94  # Given profit from the simple strategy of investing $1 in all loans
print("Profit from simple strategy (investing $1 in all loans):", simple_strategy_profit)

Total profit of an investor who invested $1 in each of the 100 selected loans: 36.38339704153835
Number of the 100 selected loans not paid back in full: 16
Profit from simple strategy (investing $1 in all loans): 20.94


Predictive models in finance often fail because they assume relationships between variables remain stable over time, which isn't true in dynamic financial markets.
To address this, analysts can retrain models frequently to ensure they reflect the latest data.
Using rolling or expanding time windows allows models to focus on more recent trends rather than outdated data.
Adding macroeconomic indicators, like interest rates, can help models adapt to broader economic changes.
Finally, stress testing and scenario analysis help prepare models for unexpected financial shifts.